In [2]:
import pyrealsense2 as rs

ctx = rs.context()
devices = ctx.query_devices()
print("num devices:", len(devices))

for i, dev in enumerate(devices):
    name = dev.get_info(rs.camera_info.name)
    sn   = dev.get_info(rs.camera_info.serial_number)
    pid  = dev.get_info(rs.camera_info.product_id)
    usb  = dev.get_info(rs.camera_info.usb_type_descriptor)
    print(f"[{i}] name={name}, sn={sn}, pid={pid}, usb={usb}")


num devices: 1
[0] name=Intel RealSense D455, sn=234322306575, pid=0B5C, usb=2.1


### 영상 촬영

In [4]:
import pyrealsense2 as rs
import numpy as np
import cv2
import os
import time

def get_next_filename(base_name="color_output", ext="mp4"):
    idx = 1
    while True:
        filename = f"{base_name}_{idx}.{ext}"
        if not os.path.exists(filename):
            return filename
        idx += 1

def pick_stream_params(usb_desc: str):
    """
    USB2(2.x)면 대역폭 문제로 1280x720@30이 실패할 수 있어
    안전하게 640x480@30으로 낮춰서 시작.
    """
    usb_desc = (usb_desc or "").strip()
    if usb_desc.startswith("2"):
        return 640, 480, 30
    return 1280, 720, 30

def main():
    ctx = rs.context()
    devs = ctx.query_devices()
    if len(devs) == 0:
        raise RuntimeError("No RealSense devices found.")

    dev = devs[0]
    name = dev.get_info(rs.camera_info.name)
    sn   = dev.get_info(rs.camera_info.serial_number)
    usb  = dev.get_info(rs.camera_info.usb_type_descriptor)

    print(f"[INFO] Device: {name} / SN: {sn} / USB: {usb}")

    width, height, fps = pick_stream_params(usb)
    print(f"[INFO] Stream request: color {width}x{height}@{fps}")

    pipeline = rs.pipeline()
    config = rs.config()

    # 특정 장치로 고정(여러 대 있을 때도 안전)
    config.enable_device(sn)
    config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)

    try:
        pipeline_profile = pipeline.start(config)
    except Exception as e:
        print("\n[ERROR] pipeline.start(config) failed.")
        print("        Likely due to unsupported stream or USB bandwidth.")
        print("        Try lower resolution/fps or use USB3 port/cable.")
        raise

    # 프리뷰 창
    win_name = "D455 Color"
    cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win_name, width, height)

    save_path = get_next_filename(base_name="color_output", ext="mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(save_path, fourcc, fps, (width, height))

    if not out.isOpened():
        raise RuntimeError(f"VideoWriter open failed: {save_path}")

    print(f"[INFO] Recording started → {save_path}")
    print("[INFO] Press Ctrl+C or press ESC to stop.\n")

    SEGMENT_DURATION = 120  # seconds
    start_time = time.time()
    dot_last = time.time()

    try:
        while True:
            frames = pipeline.wait_for_frames()
            color_frame = frames.get_color_frame()
            if not color_frame:
                continue

            color = np.asanyarray(color_frame.get_data())

            out.write(color)
            cv2.imshow(win_name, color)

            key = cv2.waitKey(1) & 0xFF
            if key == 27:
                print("\n[INFO] ESC pressed. Stopping recording...")
                break

            if time.time() - dot_last >= 0.1:
                print(".", end="", flush=True)
                dot_last = time.time()

            if time.time() - start_time >= SEGMENT_DURATION:
                out.release()
                print("\n[INFO] 2 minutes reached, starting new file...")

                save_path = get_next_filename(base_name="color_output", ext="mp4")
                out = cv2.VideoWriter(save_path, fourcc, fps, (width, height))
                if not out.isOpened():
                    raise RuntimeError(f"VideoWriter open failed: {save_path}")
                print(f"[INFO] New file → {save_path}")

                start_time = time.time()

    except KeyboardInterrupt:
        print("\n[INFO] Recording stopped by user (Ctrl+C).")

    finally:
        try:
            out.release()
        except:
            pass
        try:
            pipeline.stop()
        except:
            pass
        cv2.destroyAllWindows()
        print(f"[INFO] Last file saved: {save_path}")

if __name__ == "__main__":
    main()


[INFO] Device: Intel RealSense D455 / SN: 234322306575 / USB: 2.1
[INFO] Stream request: color 640x480@30
[INFO] Recording started → color_output_2.mp4
[INFO] Press Ctrl+C or press ESC to stop.

.[INFO] Last file saved: color_output_2.mp4


RuntimeError: Frame didn't arrive within 5000

In [2]:
import cv2
import os
import glob

base_dir = "/home/dw/ws_job_msislab/amr_project/data/video_data/20260105_video"

# 최상위 images 폴더 생성
root_output_dir = os.path.join(base_dir, "images")
os.makedirs(root_output_dir, exist_ok=True)

# MP4 파일 리스트 가져오기
video_list = sorted(glob.glob(os.path.join(base_dir, "color_output_*.mp4")))

print(f"[INFO] 감지된 MP4 파일 수: {len(video_list)}")

for video_path in video_list:
    video_name = os.path.basename(video_path).replace(".mp4", "")
    print(f"\n[START] {video_name} 변환 시작")

    # 영상별 폴더 생성
    output_dir = os.path.join(root_output_dir, video_name)
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] 비디오 열기 실패: {video_path}")
        continue

    frame_idx = 1  # 영상마다 1부터 시작

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        prefix = "20251230"
        save_name = f"{prefix}_{video_name}_{frame_idx:06d}.jpg"
        save_path = os.path.join(output_dir, save_name)
        cv2.imwrite(save_path, frame)

        frame_idx += 1

        if frame_idx % 500 == 0:
            print(f"[INFO] {video_name}: {frame_idx}장 저장 중…")

    cap.release()
    print(f"[DONE] {video_name}: 총 {frame_idx-1}장 저장")

print("\n[ALL DONE] 모든 MP4 변환 완료!")



[INFO] 감지된 MP4 파일 수: 3

[START] color_output_1 변환 시작
[INFO] color_output_1: 500장 저장 중…
[INFO] color_output_1: 1000장 저장 중…
[INFO] color_output_1: 1500장 저장 중…
[INFO] color_output_1: 2000장 저장 중…
[INFO] color_output_1: 2500장 저장 중…
[INFO] color_output_1: 3000장 저장 중…
[INFO] color_output_1: 3500장 저장 중…
[DONE] color_output_1: 총 3593장 저장

[START] color_output_2 변환 시작
[INFO] color_output_2: 500장 저장 중…
[INFO] color_output_2: 1000장 저장 중…
[INFO] color_output_2: 1500장 저장 중…
[INFO] color_output_2: 2000장 저장 중…
[INFO] color_output_2: 2500장 저장 중…
[INFO] color_output_2: 3000장 저장 중…
[INFO] color_output_2: 3500장 저장 중…
[DONE] color_output_2: 총 3600장 저장

[START] color_output_3 변환 시작
[DONE] color_output_3: 총 11장 저장

[ALL DONE] 모든 MP4 변환 완료!


### yolo_cls

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# 여러 원본 폴더들
origin_bases = [
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_cls/20251216_2",
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_cls/20251217",
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_cls/20251217_2",
]

target_base = "/home/dw/ws_job_msislab/amr_project/for_training/20251217_2"

train_dir = os.path.join(target_base, "train")
val_dir = os.path.join(target_base, "val")

# 기존 결과 삭제 후 다시 생성
shutil.rmtree(target_base, ignore_errors=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

classes = ["box_0", "box_1", "box_2", "box_3", "box_4", "unknown"]
IMG_EXTS = (".jpg", ".jpeg", ".png")

TEST_SIZE = 0.15
RANDOM_STATE = 25

def copy_with_prefix(src_path: str, dst_folder: str, prefix: str):
    """
    폴더명 prefix 붙여 저장해서 충돌 방지 + 출처 추적 가능
    """
    base = os.path.basename(src_path)
    name, ext = os.path.splitext(base)
    out_name = f"{prefix}__{name}{ext}"
    dst_path = os.path.join(dst_folder, out_name)

    # 혹시 같은 파일이 또 있으면 _1, _2 붙이기
    if not os.path.exists(dst_path):
        shutil.copy2(src_path, dst_path)
        return

    i = 1
    while True:
        out_name2 = f"{prefix}__{name}_{i}{ext}"
        dst_path2 = os.path.join(dst_folder, out_name2)
        if not os.path.exists(dst_path2):
            shutil.copy2(src_path, dst_path2)
            return
        i += 1


# 클래스별 전체 누적 카운트용
total_train = {c: 0 for c in classes}
total_val = {c: 0 for c in classes}

for origin_base in origin_bases:
    folder_tag = os.path.basename(origin_base.rstrip("/"))

    print(f"\n========== [ORIGIN] {origin_base} (tag={folder_tag}) ==========")

    for cls in classes:
        src_folder = os.path.join(origin_base, cls)

        if not os.path.isdir(src_folder):
            print(f"[WARN] 폴더 없음: {src_folder} (skip)")
            continue

        images = [
            os.path.join(src_folder, f)
            for f in os.listdir(src_folder)
            if f.lower().endswith(IMG_EXTS)
        ]

        print(f"[INFO] {folder_tag}/{cls}: 총 {len(images)}장")

        if len(images) == 0:
            continue

        # 너무 적으면 val이 0장이 될 수 있으니 예외 처리
        if len(images) < 2:
            train_imgs, val_imgs = images, []
        else:
            # 폴더별(=origin_base별) 85/15 유지 split
            train_imgs, val_imgs = train_test_split(
                images,
                test_size=TEST_SIZE,
                random_state=RANDOM_STATE,
                shuffle=True
            )

        train_cls_dir = os.path.join(train_dir, cls)
        val_cls_dir = os.path.join(val_dir, cls)
        os.makedirs(train_cls_dir, exist_ok=True)
        os.makedirs(val_cls_dir, exist_ok=True)

        for img in train_imgs:
            copy_with_prefix(img, train_cls_dir, prefix=folder_tag)
        for img in val_imgs:
            copy_with_prefix(img, val_cls_dir, prefix=folder_tag)

        total_train[cls] += len(train_imgs)
        total_val[cls] += len(val_imgs)

        print(f"[DONE] {folder_tag}/{cls} → train {len(train_imgs)}, val {len(val_imgs)}")

print("\n========== [SUMMARY] 전체 누적 ==========")
for cls in classes:
    print(f"{cls:10s} | train {total_train[cls]:6d} | val {total_val[cls]:6d}")

print("\n[ALL DONE] 폴더별 비율 유지(각 폴더 85/15) 후 합쳐서 완료!")


In [ ]:
from ultralytics import YOLO

# -------------------------------------------------
# 모델 로드 (classification)
# -------------------------------------------------
model = YOLO("yolo11s-cls.pt")

data_path = "/home/dw/ws_job_msislab/amr_project/for_training/20251217_2"

# -------------------------------------------------
# 학습
# -------------------------------------------------
model.train(
    data=data_path,
    epochs=500,
    imgsz=224,
    batch=32,
    patience=25,
    save=True,
    save_period=0,
    seed=25,
    deterministic=True,

    optimizer="AdamW",
    lr0=1e-3,
    lrf=1e-2,
    weight_decay=5e-4,
    momentum=0.9,

    cos_lr=True,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.05,
    scale=0.2,
    shear=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=0.0,
    mixup=0.1,
    copy_paste=0.0,

    erasing=0.0,          # ✅ Random Erasing(검은 네모) 끄기

    label_smoothing=0.05,
    dropout=0.2,

    val=True,
    split="val",
    save_json=False,

    device=0,
    workers=8,
    amp=True,
    cache=False,
    verbose=True,
)




### obb

In [ ]:
import os
import shutil

# =========================
# 경로 설정
# =========================
SRC_DIRS = [
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/20251210",
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/20251218",
    "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/20251230",
]

DST_DIR = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

# =========================
# 실행
# =========================
os.makedirs(DST_DIR, exist_ok=True)

copied = 0
skipped = 0

for src in SRC_DIRS:
    files = os.listdir(src)

    for fname in files:
        if not (fname.endswith(".jpg") or fname.endswith(".json")):
            continue

        src_path = os.path.join(src, fname)
        dst_path = os.path.join(DST_DIR, fname)

        if os.path.exists(dst_path):
            skipped += 1
            continue

        shutil.copy(src_path, dst_path)
        copied += 1

print("===================================")
print(f"✅ 복사 완료")
print(f" - 복사된 파일 : {copied}")
print(f" - 이미 존재해서 스킵 : {skipped}")
print(f" - 대상 폴더 : {DST_DIR}")
print("===================================")


In [ ]:
import json
import glob
import os

# 검사할 폴더
DIR = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

# 폴더 내 모든 JSON 파일 로드
json_files = sorted(glob.glob(os.path.join(DIR, "*.json")))

print("\n===== YOLO-OBB 불가 항목만 출력 (polygon 4점만 허용) =====\n")
print(f"총 JSON 파일: {len(json_files)}개\n")

if not json_files:
    print("❌ JSON 파일이 없습니다.")
    raise SystemExit(1)

total_bad = 0
files_with_bad = 0

for json_path in json_files:
    try:
        with open(json_path, "r") as f:
            data = json.load(f)
    except Exception as e:
        print(f"❌ 파일 열기 실패: {os.path.basename(json_path)}  ({e})")
        total_bad += 1
        files_with_bad += 1
        continue

    shapes = data.get("shapes", [])
    if not shapes:
        # 라벨이 없으면 굳이 불가로 보지 않을 거면 주석 처리 가능
        # print(f"❌ shapes 없음: {os.path.basename(json_path)}")
        continue

    file_bad = 0

    for idx, s in enumerate(shapes):
        if s.get("shape_type") != "polygon":
            # polygon이 아닌 건 OBB 학습 불가로 볼지 여부에 따라 출력/카운트 결정
            # 여기서는 "불가 항목"으로 출력함
            st = s.get("shape_type")
            label = s.get("label", "")
            print(f"\n📌 파일: {os.path.basename(json_path)}")
            print(f"  [{idx}] ❌ polygon 아님 → {st} (label={label})")
            file_bad += 1
            continue

        pts = s.get("points", [])
        n = len(pts)
        if n != 4:
            label = s.get("label", "")
            print(f"\n📌 파일: {os.path.basename(json_path)}")
            print(f"  [{idx}] ❌ {n}점 polygon → YOLO-OBB 불가 (label={label})")
            file_bad += 1

    if file_bad > 0:
        total_bad += file_bad
        files_with_bad += 1

print("\n===== 요약 =====")
print(f"불가 항목 총 개수: {total_bad}개")
print(f"불가 항목 포함 파일: {files_with_bad}개\n")


In [ ]:
import json
import os
from glob import glob
import numpy as np

# 변환할 폴더
BASE = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

json_files = sorted(glob(os.path.join(BASE, "*.json")))

print("\n===== JSON → YOLO11-OBB 변환 시작 =====\n")


def sort_polygon_points(pts):
    """
    4개 점을 중심 기준으로 시계/반시계 순서로 정렬.
    YOLO OBB 학습을 위한 필수 단계.
    """

    pts = np.array(pts)

    # 중심점
    cx = np.mean(pts[:, 0])
    cy = np.mean(pts[:, 1])

    # 각도 계산
    angles = np.arctan2(pts[:, 1] - cy, pts[:, 0] - cx)

    # 각도 기준 정렬 (반시계 방향)
    sorted_idx = np.argsort(angles)

    return pts[sorted_idx].tolist()



for json_path in json_files:
    with open(json_path, "r") as f:
        data = json.load(f)

    img_w = data["imageWidth"]
    img_h = data["imageHeight"]

    shapes = data.get("shapes", [])

    # 출력 txt 경로
    txt_path = json_path.replace(".json", ".txt")
    lines = []

    for s in shapes:
        if s.get("shape_type") != "polygon":
            continue

        pts = s.get("points", [])

        if len(pts) != 4:
            print(f"❌ {json_path} — polygon 점 {len(pts)}개 → 4점 필수")
            continue

        # 점 순서 정렬
        pts_sorted = sort_polygon_points(pts)

        # YOLO 정규화 좌표
        yolo_pts = []
        for (x, y) in pts_sorted:
            yolo_pts.append(x / img_w)
            yolo_pts.append(y / img_h)

        # class_id = 0
        line = "0 " + " ".join([f"{v:.6f}" for v in yolo_pts])
        lines.append(line)

    # txt 파일 저장
    with open(txt_path, "w") as f:
        for line in lines:
            f.write(line + "\n")

    print(f"[OK] {os.path.basename(txt_path)} 생성됨")

print("\n===== 변환 완료 =====\n")



In [ ]:
import os
from glob import glob

BASE = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

print("\n===== JSON 파일 삭제 =====")
json_files = sorted(glob(os.path.join(BASE, "*.json")))

if not json_files:
    print("✅ 삭제할 JSON 파일이 없습니다.")
else:
    for jf in json_files:
        try:
            os.remove(jf)
            print(f"삭제됨 → {os.path.basename(jf)}")
        except Exception as e:
            print(f"❌ 삭제 실패 → {os.path.basename(jf)} ({e})")

print("\n===== 완료 =====\n")


In [ ]:
import os
import shutil
from glob import glob

DIR = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

# ✅ 원본 1개당 몇 장을 "추가로" 만들지 (예: 2면 원본 + 2개 복사 = 총 3장 느낌)
COPIES_PER_ORIGINAL = 1  # <- 여기만 바꿔

# 이미지 목록
imgs = sorted(glob(os.path.join(DIR, "*.jpg")))

if not imgs:
    raise SystemExit("❌ jpg 파일이 없습니다.")

made = 0
skipped = 0

for img_path in imgs:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(DIR, stem + ".txt")

    if not os.path.exists(lbl_path):
        print(f"⚠️ 라벨 없음 -> 스킵: {stem}.jpg")
        skipped += 1
        continue

    for k in range(1, COPIES_PER_ORIGINAL + 1):
        new_stem = f"{stem}__copy{k:02d}"
        new_img = os.path.join(DIR, new_stem + ".jpg")
        new_lbl = os.path.join(DIR, new_stem + ".txt")

        # 혹시 이미 존재하면 덮어쓰기 위험 -> 스킵
        if os.path.exists(new_img) or os.path.exists(new_lbl):
            print(f"⚠️ 이미 존재 -> 스킵: {new_stem}")
            continue

        shutil.copy2(img_path, new_img)
        shutil.copy2(lbl_path, new_lbl)
        made += 1

print("\n===== 완료 =====")
print(f"복사 생성 쌍: {made}개")
print(f"라벨 누락 스킵: {skipped}개")
print(f"대상 폴더: {DIR}\n")


In [ ]:
import os
import shutil
from glob import glob
import random
import yaml

BASE = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"

# -----------------------------
# Config
# -----------------------------
VAL_RATIO = 0.2
SEED = 0          # 재현성 필요 없으면 None로 두거나 이 줄 삭제해도 됨
NC = 1
NAMES = ["box"]

# -----------------------------
# 1) train/val/images/labels 디렉토리 생성
# -----------------------------
train_img_dir = os.path.join(BASE, "train/images")
train_lbl_dir = os.path.join(BASE, "train/labels")
val_img_dir   = os.path.join(BASE, "val/images")
val_lbl_dir   = os.path.join(BASE, "val/labels")

for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

# -----------------------------
# 2) 이미지 + 라벨 파일 수집
# -----------------------------
print("\n===== Step 2: 이미지 + 라벨 파일 수집 =====")
images = sorted(glob(os.path.join(BASE, "*.jpg")))
labels = sorted(glob(os.path.join(BASE, "*.txt")))

print(f"이미지: {len(images)}개")
print(f"라벨:  {len(labels)}개")

assert len(images) == len(labels), "\n❌ 이미지/라벨 개수가 다릅니다!"

pairs = list(zip(images, labels))

if SEED is not None:
    random.seed(SEED)
random.shuffle(pairs)

val_count = int(len(pairs) * VAL_RATIO)
val_set = pairs[:val_count]
train_set = pairs[val_count:]

print(f"Train: {len(train_set)}개")
print(f"Val:   {len(val_set)}개")

# -----------------------------
# 3) 파일 이동 (train/val)
# -----------------------------
def move_pairs(pairs_, img_dest, lbl_dest):
    for img, lbl in pairs_:
        shutil.move(img, os.path.join(img_dest, os.path.basename(img)))
        shutil.move(lbl, os.path.join(lbl_dest, os.path.basename(lbl)))

print("\n===== Step 3: 파일 이동 (train/val) =====")
move_pairs(train_set, train_img_dir, train_lbl_dir)
move_pairs(val_set, val_img_dir, val_lbl_dir)

# -----------------------------
# 4) data.yaml 생성
# -----------------------------
print("\n===== Step 4: data.yaml 생성 =====")
yaml_path = os.path.join(BASE, "data.yaml")

data_yaml = {
    "train": "train/images",
    "val": "val/images",
    "nc": NC,
    "names": NAMES,
}

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml 생성됨 → {yaml_path}")
print("\n===== ✔ 분할 + yaml 생성 완료 =====\n")


In [ ]:
import os
import shutil
from glob import glob
import random
import cv2
import numpy as np

BASE = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb/train"
IMG_DIR = os.path.join(BASE, "images")
LBL_DIR = os.path.join(BASE, "labels")

# -----------------------------
# 너가 조절할 값들
# -----------------------------
TARGET_GLOB = "*.jpg"   # 복사본만 증강. 원본까지면 "*.jpg"
AUG_PER_IMAGE = 2             # 2~3 추천 (원하는 만큼)
SEED = 0                      # 재현성 필요 없으면 None

BRIGHTNESS_DELTA = 18
CONTRAST_RANGE = (0.88, 1.12)
GAMMA_RANGE = (0.90, 1.15)
HSV_H = 3
HSV_S = 18
HSV_V = 18

NOISE_STD_RANGE = (0, 6)
BLUR_K_CHOICES = [0, 3, 5]    # 0이면 블러 없음

# -----------------------------
# Photometric ops (NO SHADOW, NO JPEG RECOMPRESS)
# -----------------------------
def apply_photometric(img_bgr, py_rng: random.Random, np_rng: np.random.Generator):
    out = img_bgr.copy().astype(np.float32)

    # brightness / contrast
    alpha = py_rng.uniform(*CONTRAST_RANGE)
    beta = py_rng.uniform(-BRIGHTNESS_DELTA, BRIGHTNESS_DELTA)
    out = out * alpha + beta

    # gamma
    out_u8 = np.clip(out, 0, 255).astype(np.uint8)
    gamma = py_rng.uniform(*GAMMA_RANGE)
    lut = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)], dtype=np.uint8)
    out_u8 = cv2.LUT(out_u8, lut)

    # HSV jitter
    hsv = cv2.cvtColor(out_u8, cv2.COLOR_BGR2HSV).astype(np.int16)
    hsv[..., 0] = (hsv[..., 0] + py_rng.randint(-HSV_H, HSV_H + 1)) % 180
    hsv[..., 1] = np.clip(hsv[..., 1] + py_rng.randint(-HSV_S, HSV_S + 1), 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] + py_rng.randint(-HSV_V, HSV_V + 1), 0, 255)
    out_u8 = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

    # noise
    std = py_rng.uniform(*NOISE_STD_RANGE)
    if std > 0:
        noise = np_rng.normal(0.0, std, size=out_u8.shape).astype(np.float32)
        out_u8 = np.clip(out_u8.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # blur
    k = py_rng.choice(BLUR_K_CHOICES)
    if k and k >= 3:
        out_u8 = cv2.GaussianBlur(out_u8, (k, k), 0)

    return out_u8

# -----------------------------
# Main
# -----------------------------
py_rng = random.Random(SEED) if SEED is not None else random.Random()
np_rng = np.random.default_rng(SEED)

imgs = sorted(glob(os.path.join(IMG_DIR, TARGET_GLOB)))
if not imgs:
    raise SystemExit(f"❌ 대상 이미지가 없습니다: {os.path.join(IMG_DIR, TARGET_GLOB)}")

made = 0
skipped = 0

print(f"\nIMG_DIR: {IMG_DIR}")
print(f"LBL_DIR: {LBL_DIR}")
print(f"TARGET: {TARGET_GLOB}")
print(f"AUG_PER_IMAGE: {AUG_PER_IMAGE}\n")

for img_path in imgs:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    txt_path = os.path.join(LBL_DIR, stem + ".txt")

    if not os.path.exists(txt_path):
        print(f"⚠️ 라벨 없음 -> 스킵: {stem}")
        skipped += 1
        continue

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ 이미지 로드 실패 -> 스킵: {stem}")
        skipped += 1
        continue

    for k in range(1, AUG_PER_IMAGE + 1):
        out_stem = f"{stem}__ph{k:02d}"
        out_img = os.path.join(IMG_DIR, out_stem + ".jpg")
        out_txt = os.path.join(LBL_DIR, out_stem + ".txt")

        # 충돌 방지
        if os.path.exists(out_img) or os.path.exists(out_txt):
            out_stem = f"{stem}__ph{k:02d}_{py_rng.randint(1000,9999)}"
            out_img = os.path.join(IMG_DIR, out_stem + ".jpg")
            out_txt = os.path.join(LBL_DIR, out_stem + ".txt")

        img2 = apply_photometric(img, py_rng, np_rng)
        cv2.imwrite(out_img, img2)
        shutil.copy2(txt_path, out_txt)

        made += 1

print("\n===== 완료 =====")
print(f"증강 생성 쌍: {made}개")
print(f"스킵: {skipped}개\n")


### obb 맞나 점검

In [ ]:
import os
import cv2
import numpy as np
from glob import glob

BASE = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb"
IMG_DIR = os.path.join(BASE, "train/images")
LBL_DIR = os.path.join(BASE, "train/labels")
OUT_DIR = os.path.join(BASE, "debug_overlays")
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# Collect images
# -------------------------
img_list = []
for ext in ("*.jpg", "*.JPG", "*.png", "*.PNG", "*.jpeg", "*.JPEG"):
    img_list += sorted(glob(os.path.join(IMG_DIR, ext)))

if not img_list:
    raise FileNotFoundError(f"No images found in: {IMG_DIR}")

def clip_pt(x, y, w, h):
    x = max(0, min(w - 1, int(round(x))))
    y = max(0, min(h - 1, int(round(y))))
    return x, y

ok_cnt = 0
skip_no_label = 0
skip_bad_label = 0
empty_label = 0
read_fail = 0
write_fail = 0

print(f"[INFO] images: {len(img_list)}")
print(f"[INFO] labels dir: {LBL_DIR}")
print(f"[INFO] out dir: {OUT_DIR}")

for idx, img_path in enumerate(img_list, 1):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(LBL_DIR, stem + ".txt")

    if not os.path.exists(lbl_path):
        skip_no_label += 1
        continue

    img = cv2.imread(img_path)
    if img is None:
        read_fail += 1
        print(f"[READ_FAIL] {img_path}")
        continue

    h, w = img.shape[:2]

    with open(lbl_path, "r") as f:
        lines = [ln.strip() for ln in f.readlines() if ln.strip()]

    if not lines:
        empty_label += 1
        # 그래도 “빈 라벨” 표시해서 저장(원하면 아래 저장 부분 주석 처리 가능)
        cv2.putText(img, "EMPTY_LABEL", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    drawn = 0
    bad = False

    for li, ln in enumerate(lines):
        parts = ln.split()
        if len(parts) != 9:
            bad = True
            print(f"[BAD_LABEL] {stem}.txt line{li}: tokens={len(parts)} -> {ln}")
            continue

        cls = int(float(parts[0]))
        vals = list(map(float, parts[1:]))

        pts = []
        for i in range(0, 8, 2):
            x = vals[i] * w
            y = vals[i + 1] * h
            pts.append(clip_pt(x, y, w, h))

        pts_np = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)

        # polygon
        cv2.polylines(img, [pts_np], isClosed=True, color=(255, 0, 0), thickness=2)

        # point index 0~3
        for pi, (x, y) in enumerate(pts):
            cv2.circle(img, (x, y), 4, (0, 255, 255), -1)
            cv2.putText(img, f"{pi}", (x + 6, y - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        # class text near point0
        x0, y0 = pts[0]
        cv2.putText(img, f"cls={cls}", (x0 + 8, y0 + 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        drawn += 1

    if bad and drawn == 0:
        skip_bad_label += 1
        continue

    out_path = os.path.join(OUT_DIR, f"{stem}__overlay.jpg")
    ok = cv2.imwrite(out_path, img)
    if not ok:
        write_fail += 1
        print(f"[WRITE_FAIL] {out_path}")
        continue

    ok_cnt += 1

    if idx % 200 == 0 or idx == len(img_list):
        print(f"[PROGRESS] {idx}/{len(img_list)}  saved={ok_cnt}  no_label={skip_no_label}  bad={skip_bad_label}  empty={empty_label}")

print("\n===== DONE =====")
print("saved:", ok_cnt)
print("skip_no_label:", skip_no_label)
print("skip_bad_label:", skip_bad_label)
print("empty_label:", empty_label)
print("read_fail:", read_fail)
print("write_fail:", write_fail)
print("out_dir:", OUT_DIR)


In [1]:
from ultralytics import YOLO

# ----------------------------------------------------
# 1) YOLO11S-OBB 모델 로드
# ----------------------------------------------------
model = YOLO("yolo11s-obb.pt")   # OBB 전용 pretrained 모델

# ----------------------------------------------------
# 2) DATA.YAML 경로
# ----------------------------------------------------
DATA_PATH = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb/data.yaml"

# ----------------------------------------------------
# 3) 학습 실행
# ----------------------------------------------------
model.train(
    # ---------- 필수 ----------
    data=DATA_PATH,
    epochs=500,                # 스모크 테스트
    imgsz=640,
    batch=4,
    device=0,
    workers=2,

    # ---------- 실험 관리 ----------
    project="../runs/obb",      # runs/obb 아래로 정리
    name="20251231_obb_test",    # 실험 이름
    exist_ok=True,           # 같은 이름 덮어쓰기 허용
    seed=25,                 # 재현성 확보
    deterministic=True,

    # ---------- Optimizer / LR ----------
    optimizer="AdamW",       # OBB에서 비교적 안정적
    lr0=1e-3,                # 초기 learning rate
    lrf=0.01,                # 최종 LR 비율
    cos_lr=True,             # cosine scheduler

    # ---------- 학습 안정성 ----------
    patience=50,             # early stopping (스모크라 사실상 의미 없음)
    warmup_epochs=1,
    amp=True,                # mixed precision (GPU 메모리 절약)

    # ---------- 데이터 로딩 ----------
    cache="ram",             # 데이터 적으면 RAM 캐시 추천
    rect=False,              # OBB는 보통 False 권장

    # ---------- Augmentation (OBB 기준 보수적으로) ----------
    hsv_h=0.015,
    hsv_s=0.2,
    hsv_v=0.2,
    degrees=0,           # 회전은 OBB 특성상 소량만
    translate=0,
    scale=0,
    shear=0,
    flipud=0,
    fliplr=0.2,
    mosaic=0,
    mixup=0,

    # ---------- 저장 / 로그 ----------
    save=True,
    save_period=50,           # epoch마다 weight 저장
    plots=True,              # loss, metrics plot 생성
    verbose=True,
)

print("\n🎉 Training Finished! Check the 'runs/obb' folder.")



New https://pypi.org/project/ultralytics/8.3.245 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.239 🚀 Python-3.10.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Ti, 7932MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb/data.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=True, fliplr=0.2, flipud=0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.2, hsv_v=0.2, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0, mode=train, model=yolo11s-obb.pt, momentum=0.937, mosaic=0, multi_scal

In [ ]:
from ultralytics import YOLO

# ----------------------------------------------------
# 1) YOLO11S-SEG 모델 로드 (Segmentation 전용)
# ----------------------------------------------------
model = YOLO("yolo11s-seg.pt")   # .pt 파일명을 -seg로 변경

# ----------------------------------------------------
# 2) DATA.YAML 경로 (동일한 데이터 사용)
# ----------------------------------------------------
DATA_PATH = "/home/dw/ws_job_msislab/amr_project/for_training/20251231_obb/data.yaml"

# ----------------------------------------------------
# 3) 학습 실행
# ----------------------------------------------------
model.train(
    # ---------- 필수 ----------
    data=DATA_PATH,
    epochs=500,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,

    # ---------- 실험 관리 ----------
    project="../runs/seg",      # runs/seg 폴더로 구분하여 저장
    name="20251231_seg_test",   
    exist_ok=True,
    seed=25,
    deterministic=True,

    # ---------- Optimizer / LR ----------
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,

    # ---------- 학습 안정성 ----------
    patience=50,             # Seg 모델은 조금 더 긴 호흡이 필요할 수 있어 30 권장
    warmup_epochs=1,
    amp=True,

    # ---------- 데이터 로딩 ----------
    cache="ram",
    rect=False,

    # ---------- Augmentation ----------
    # Seg는 OBB보다 이미지 변형에 강하므로 기본 설정을 활용해도 좋습니다.
    hsv_h=0.015,
    hsv_s=0.2,
    hsv_v=0.2,
    degrees=0,            # Seg는 약간의 회전을 줘도 잘 학습합니다.
    translate=0,
    scale=0,
    shear=0.0,
    flipud=0.0,
    fliplr=0.2,
    mosaic=0,              # 물체가 작거나 구석에 있다면 Mosaic 켜는 것 추천
    mixup=0.0,

    # ---------- 저장 / 로그 ----------
    save=True,
    save_period=50,
    plots=True,
    verbose=True,
)

print("\n🎉 Segmentation Training Finished! Check the 'runs/seg' folder.")

In [1]:
from ultralytics import YOLO
import os

# 1. 모델 및 경로 설정
model_path = "/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb/20251231_obb_test/weights/last.pt"
source_dir = "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기"
project_path = "/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/obb" # 결과가 저장될 메인 폴더

# 2. 모델 로드
model = YOLO(model_path)

# 3. 추론 실행 (결과는 project/name 폴더 안에 저장됨)
results = model.predict(
    source=source_dir,
    project=project_path,
    name="viz_results",     # 저장될 폴더 이름
    save=True,              # 이미지 저장 활성화
    conf=0.75,              # 신뢰도 임계값 (상황에 따라 조절)
    device=0,               # GPU 사용
    exist_ok=True           # 폴더가 이미 있으면 덮어쓰기/안에 저장
)

print(f"✅ 시각화 완료! 결과는 다음 경로에서 확인하세요: {os.path.join(project_path, 'viz_results')}")


image 1/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000758.jpg: 384x640 3 boxs, 41.3ms
image 2/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000784.jpg: 384x640 3 boxs, 3.8ms
image 3/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000785.jpg: 384x640 3 boxs, 4.3ms
image 4/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000786.jpg: 384x640 3 boxs, 3.9ms
image 5/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000938.jpg: 384x640 2 boxs, 3.9ms
image 6/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000948.jpg: 384x640 2 boxs, 3.9ms
image 7/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000949.jpg: 384x640 2 boxs, 4.3ms
image 8/98 /home/dw/ws_job_msislab/amr_

In [2]:
from ultralytics import YOLO
import os

# 1. 모델 및 경로 설정
model_path = "/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/seg/20251231_seg_test/weights/best.pt"
source_dir = "/home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기"
project_path = "/home/dw/ws_job_msislab/amr_project/src/job_pc/runs/seg"  # seg 결과 폴더 권장

# 2. 모델 로드
model = YOLO(model_path)

# 3. 추론 실행
results = model.predict(
    source=source_dir,
    project=project_path,
    name="viz_results_seg",  # 저장될 폴더 이름
    save=True,               # 이미지 저장
    conf=0.75,               # seg도 conf 적용됨(마스크/박스 필터)
    device=0,
    exist_ok=True
)

out_dir = os.path.join(project_path, "viz_results_seg")
print(f"✅ 시각화 완료! 결과는 다음 경로에서 확인하세요: {out_dir}")



image 1/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000758.jpg: 384x640 3 boxs, 9.3ms
image 2/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000784.jpg: 384x640 3 boxs, 4.2ms
image 3/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000785.jpg: 384x640 3 boxs, 3.9ms
image 4/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000786.jpg: 384x640 3 boxs, 4.1ms
image 5/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000938.jpg: 384x640 3 boxs, 3.9ms
image 6/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000948.jpg: 384x640 3 boxs, 4.1ms
image 7/98 /home/dw/ws_job_msislab/amr_project/data/image_data/for_yolo_obj/대기/20251210_color_output_2_000949.jpg: 384x640 3 boxs, 4.1ms
image 8/98 /home/dw/ws_job_msislab/amr_p

### yolo_pose